# التنبؤ باحتمال إعادة إدخال المريض إلى المستشفى خلال 30 يومًا

**المشروع:** Readmission Risk Prediction  
**نوع التعلم:** تعلم آلي مُراقَب (Supervised Learning — سوبرڤايزد ليرننغ)  
**المهمة:** تصنيف ثنائي (Binary Classification — باينري كلاسيفيكيشن)

## هدف المشروع
استخدام سجلات مستشفى تاريخية لتدريب نماذج تتوقع ما إذا كان سجل المريض مرتبطًا بإعادة إدخال خلال أقل من 30 يومًا.

**تعريف الهدف في هذا المشروع:**
- `1`: قيمة `readmitted` الأصلية تساوي `<30`.
- `0`: قيمة `readmitted` تساوي `NO` أو `>30`.

هذا تعريف تحليلي للبيانات وليس تشخيصًا طبيًا أو توصية علاجية. البيانات من مستشفيات أمريكية قديمة، ولذلك لا ينبغي اعتبار النتائج صالحة تلقائيًا لمستشفيات اليمن أو لقرارات سريرية حقيقية.

**مصدر البيانات:** UCI Machine Learning Repository — Diabetes 130-US Hospitals for Years 1999–2008  
https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

> ملاحظة: السجلات قد تتكرر للمريض نفسه. لتقليل تسرب البيانات، نقسم البيانات بحسب `patient_nbr` بحيث لا يظهر المريض نفسه في التدريب والاختبار معًا.

## 1. تثبيت واستيراد المكتبات

المكتبات الأساسية:
- **pandas (بانْدَز):** قراءة البيانات وتنظيمها.
- **matplotlib (ماتبلوتليب):** رسم المخططات.
- **scikit-learn (ساي كِت ليرن):** تجهيز البيانات وتدريب النماذج وتقييمها.

In [ ]:
import sys
import zipfile
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, RocCurveDisplay
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

print("Python:", sys.version.split()[0])
print("تم استيراد المكتبات بنجاح.")

## 2. تحميل البيانات

إذا كان الملف `diabetes_dataset.zip` موجودًا بجانب الـ Notebook، سيحاول الكود استخراج ملف `diabetic_data.csv` تلقائيًا. إذا كنتِ تستخدمين Kaggle أو Colab، ارفعي ملف ZIP أو ملف CSV إلى بيئة العمل أولًا.

In [ ]:
data_dir = Path(".")
csv_path = data_dir / "diabetic_data.csv"
zip_path = data_dir / "diabetes_dataset.zip"

if not csv_path.exists() and zip_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        target_name = next(
            (name for name in zf.namelist() if name.endswith("diabetic_data.csv")),
            None
        )
        if target_name is None:
            raise FileNotFoundError("لم أجد diabetic_data.csv داخل ملف ZIP.")
        with zf.open(target_name) as src, open(csv_path, "wb") as dst:
            dst.write(src.read())

if not csv_path.exists():
    raise FileNotFoundError(
        "لم أجد diabetic_data.csv. ضعي diabetic_data.csv أو diabetes_dataset.zip بجانب الـ Notebook."
    )

df = pd.read_csv(csv_path)
print("حجم البيانات:", df.shape)
display(df.head())

## 3. فهم الأعمدة وجودة البيانات

- **السجل (Record — ريكورد):** صف واحد يمثل دخولًا للمستشفى.
- **الخاصية (Feature — فيتشر):** معلومة نستخدمها للتنبؤ.
- **الهدف (Target — تارغت):** النتيجة التي نريد أن يتعلم النموذج توقعها.
- `patient_nbr` معرّف المريض، وليس خاصية تنبؤ؛ نستخدمه فقط لتقسيم البيانات حسب المريض.

In [ ]:
print("عدد الصفوف والأعمدة:", df.shape)
print("\nأسماء الأعمدة:")
print(df.columns.tolist())

print("\nأول نظرة على أنواع الأعمدة:")
display(df.dtypes.value_counts().rename_axis("نوع البيانات").to_frame("عدد الأعمدة"))

print("\nعدد القيم المفقودة لكل عمود (أعلى 15):")
missing_summary = df.isin(["?", ""]).sum().add(df.isna().sum()).sort_values(ascending=False)
display(missing_summary.head(15).to_frame("عدد القيم المفقودة/غير المعروفة"))

print("\nعدد السجلات المكررة بالكامل:", df.duplicated().sum())
print("\nتوزيع readmitted الأصلي:")
display(df["readmitted"].value_counts(dropna=False).to_frame("عدد السجلات"))

## 4. إنشاء الهدف الذي سنتنبأ به

سننشئ العمود `target_readmit_30`:
- `1` إذا كانت إعادة الإدخال أقل من 30 يومًا.
- `0` إذا كانت `NO` أو `>30`.

هذا يعني أننا نجمع فئتين مختلفتين في الفئة السلبية. سنذكر ذلك بوضوح في تقرير المشروع.

In [ ]:
df = df[df["readmitted"].isin(["NO", ">30", "<30"])].copy()
df["target_readmit_30"] = (df["readmitted"] == "<30").astype(int)

target_counts = df["target_readmit_30"].value_counts().sort_index()
target_names = {0: "لا إعادة إدخال خلال أقل من 30 يومًا", 1: "إعادة إدخال خلال أقل من 30 يومًا"}
display(pd.DataFrame({
    "عدد السجلات": target_counts,
    "النسبة المئوية": (target_counts / len(df) * 100).round(2)
}).rename(index=target_names))

ax = df["target_readmit_30"].value_counts().sort_index().plot(
    kind="bar", rot=0, figsize=(7, 4), title="توزيع الهدف"
)
ax.set_xlabel("الهدف: 0 أو 1")
ax.set_ylabel("عدد السجلات")
plt.tight_layout()
plt.show()

## 5. تجهيز الخصائص

سنستبعد:
- `readmitted`: لأنه مصدر إنشاء الهدف.
- `target_readmit_30`: الهدف نفسه.
- `encounter_id` و`patient_nbr`: معرّفات وليست خصائص عامة مناسبة للتنبؤ.
- `weight` و`payer_code` و`medical_specialty`: تُستبعد في هذه النسخة الأساسية بسبب ارتفاع نسبة القيم غير المتاحة أو صعوبة تفسيرها.
- `examide` و`citoglipton`: أعمدة غير مفيدة عمليًا في هذه البيانات.

القيمة `?` تعني أن المعلومة غير معروفة؛ سنحوّلها إلى قيمة مفقودة ثم نعالجها داخل Pipeline. **Pipeline (بايبلاين)** تعني ربط خطوات التجهيز والتدريب معًا حتى نتعلم قواعد المعالجة من بيانات التدريب فقط.

In [ ]:
drop_cols = [
    "readmitted", "target_readmit_30",
    "encounter_id", "patient_nbr",
    "weight", "payer_code", "medical_specialty",
    "examide", "citoglipton"
]
drop_cols = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=drop_cols).copy()
y = df["target_readmit_30"].copy()
groups = df["patient_nbr"].copy()

X = X.replace("?", np.nan)
X = X.replace("", np.nan)

# أعمدة أرقام معرّفة في قاموس البيانات كفئات، لا كقيم رقمية مستمرة.
id_like_categorical = [
    c for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
    if c in X.columns
]
for c in id_like_categorical:
    X[c] = X[c].astype("string")

# أي عمود نصي يعامل كفئة؛ الأعمدة العددية تبقى رقمية.
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

print("عدد الخصائص الكلي:", X.shape[1])
print("عدد الخصائص الرقمية:", len(numeric_features))
print("عدد الخصائص الفئوية:", len(categorical_features))
display(X.head())

## 6. تقسيم البيانات مع منع تسرب المرضى

**تسرب البيانات (Data Leakage — داتا ليكِج)** يحدث عندما يحصل النموذج أثناء التدريب على معلومات تجعل الاختبار أسهل بصورة غير واقعية.

قد يحتوي المريض نفسه على أكثر من زيارة؛ لذلك نستخدم `GroupShuffleSplit` على أساس `patient_nbr`. بهذا لا يتوزع المريض نفسه بين التدريب والاختبار.

لا نستخدم `patient_nbr` كخاصية يتعلم منها النموذج.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

print("حجم التدريب:", X_train.shape)
print("حجم الاختبار:", X_test.shape)
print("نسبة الفئة الإيجابية في التدريب:", round(y_train.mean() * 100, 2), "%")
print("نسبة الفئة الإيجابية في الاختبار:", round(y_test.mean() * 100, 2), "%")
print("هل يوجد مرضى مشتركون بين التقسيمين؟", not set(groups_train).isdisjoint(set(groups_test)))

## 7. تجهيز Pipeline

- القيم الرقمية المفقودة: نعوضها بوسيط العمود.
- القيم الفئوية المفقودة: نعوضها بأكثر قيمة تكرارًا.
- `OneHotEncoder` يحول الفئات مثل `gender` إلى أعمدة رقمية يستطيع النموذج استخدامها.
- نستخدم `StandardScaler` مع الانحدار اللوجستي فقط، لأنه يستفيد من توحيد مقياس الخصائص.

In [ ]:
numeric_preprocess_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_preprocess_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
])

preprocess_logistic = ColumnTransformer(transformers=[
    ("num", numeric_preprocess_scaled, numeric_features),
    ("cat", categorical_preprocess, categorical_features)
])

preprocess_tree = ColumnTransformer(transformers=[
    ("num", numeric_preprocess_tree, numeric_features),
    ("cat", categorical_preprocess, categorical_features)
])

print("تم إعداد خطوات تجهيز البيانات.")

## 8. تدريب النماذج

سنقارن ثلاثة نماذج:
1. **Dummy Classifier:** نموذج مرجعي بسيط يوضح إن كان النموذج المتعلم أفضل من تخمين أساسي.
2. **Logistic Regression:** نموذج تصنيف بسيط نسبيًا وسهل الشرح.
3. **Random Forest:** مجموعة أشجار قرار تعمل معًا.

`class_weight="balanced"` يجعل التدريب يراعي عدم توازن الفئات، لكنه لا يضمن أن كل مقياس سيتحسن. نقيّم الأداء على مجموعة الاختبار التي لم تُستخدم في التدريب.

In [ ]:
models = {
    "Baseline - Most Frequent": Pipeline(steps=[
        ("prep", preprocess_tree),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]),
    "Logistic Regression": Pipeline(steps=[
        ("prep", preprocess_logistic),
        ("model", LogisticRegression(
            max_iter=300,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        ))
    ]),
    "Random Forest": Pipeline(steps=[
        ("prep", preprocess_tree),
        ("model", RandomForestClassifier(
            n_estimators=120,
            max_depth=18,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42
        ))
    ])
}

results = []
fitted_models = {}

for name, model in models.items():
    print(f"جاري تدريب: {name}")
    model.fit(X_train, y_train)
    fitted_models[name] = model

    pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, prob) if len(np.unique(y_test)) == 2 else np.nan
    else:
        prob, auc = None, np.nan

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": auc
    })

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df.round(4))

## 9. فهم مقاييس التقييم

- **Accuracy (أكيورَسي):** نسبة كل التوقعات الصحيحة. قد تكون مضللة إذا كانت إحدى الفئات أكثر شيوعًا.
- **Precision (بريسيجن):** من الحالات التي توقعنا أنها ستُعاد خلال 30 يومًا، كم حالة كانت إيجابية فعلًا؟
- **Recall (ريكول):** من الحالات الإيجابية الفعلية، كم حالة اكتشفها النموذج؟
- **F1-score (إف ون):** مقياس يجمع Precision وRecall.
- **ROC-AUC:** يقيس قدرة النموذج على ترتيب الحالات الإيجابية أعلى من السلبية عبر عتبات مختلفة.

في هذا المثال التعليمي، نهتم بـ Recall للفئة الإيجابية، لكن رفعه قد يزيد الإنذارات الإيجابية الخاطئة. اختيار عتبة قرار حقيقية يتطلب مراجعة مختصين وتقييمًا سريريًا مستقلًا.

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
best_pred = best_model.predict(X_test)

print("النموذج الأعلى وفق F1 في هذه التجربة:", best_name)
print("\nتقرير التصنيف:")
print(classification_report(
    y_test, best_pred,
    target_names=["0: لا إعادة إدخال خلال أقل من 30 يومًا", "1: إعادة إدخال خلال أقل من 30 يومًا"],
    zero_division=0
))

ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred,
    display_labels=["الفئة 0", "الفئة 1"],
    values_format="d"
)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

## 10. مقارنة ROC لكل نموذج يتيح احتمالات

ROC (آر أو سي) يوضح العلاقة بين معدل اكتشاف الحالات الإيجابية والإنذارات الخاطئة عبر عتبات مختلفة.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in fitted_models.items():
    if hasattr(model, "predict_proba") and name != "Baseline - Most Frequent":
        RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", label="تخمين عشوائي")
ax.set_title("ROC Curves")
plt.tight_layout()
plt.show()

## 11. أهمية الخصائص في نموذج Random Forest

يعرض الرسم أهمية الخصائص كما يقدرها النموذج. هذه الأهمية **لا تثبت السببية** ولا تعني أن تغيير الخاصية سيغير نتيجة المريض.

In [ ]:
rf_model = fitted_models.get("Random Forest")
if rf_model is not None:
    fitted_preprocessor = rf_model.named_steps["prep"]
    rf_estimator = rf_model.named_steps["model"]
    try:
        feature_names = fitted_preprocessor.get_feature_names_out()
        importances = pd.Series(rf_estimator.feature_importances_, index=feature_names)
        top_importances = importances.sort_values(ascending=False).head(20).sort_values()
        ax = top_importances.plot(kind="barh", figsize=(9, 7), title="أهم 20 خاصية في Random Forest")
        ax.set_xlabel("Feature Importance")
        plt.tight_layout()
        plt.show()
        display(top_importances.sort_values(ascending=False).to_frame("الأهمية"))
    except Exception as exc:
        print("تعذر استخراج أسماء الخصائص في هذه البيئة:", exc)

## 12. تجربة توقع لسجل من مجموعة الاختبار

هذه تجربة تقنية على سجل موجود في مجموعة الاختبار وليست توصية طبية لمريض حقيقي. الاحتمال المعروض هو تقدير النموذج وفق البيانات والتجهيز الحاليين فقط.

In [ ]:
sample = X_test.iloc[[0]]
sample_probability = best_model.predict_proba(sample)[0, 1]
sample_prediction = int(sample_probability >= 0.5)

print("توقع النموذج:", sample_prediction)
print("الاحتمال المقدّر عند عتبة 0.5:", round(float(sample_probability), 4))
display(sample)

## 13. الخلاصة وحدود المشروع

قبل تقديم المشروع:
1. اذكري أن البيانات تاريخية ومن مستشفيات أمريكية بين 1999 و2008.
2. اذكري أن الهدف جرى تعريفه من `readmitted` وأن الفئة 0 تجمع `NO` و`>30`.
3. اذكري أن تقسيم البيانات جرى بحسب المريض لتقليل تسرب السجلات المتكررة.
4. لا تعرضي Accuracy وحدها؛ اعرضي Precision وRecall وF1 وROC-AUC ومصفوفة الالتباس.
5. لا تستخدمي هذا النموذج لتشخيص المرضى أو اتخاذ قرار علاجي. إنه مشروع تعليمي يحتاج إلى تحقق مستقل قبل أي استخدام واقعي.

## مراجع
- UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008
- Strack, B. et al. (2014), Impact of HbA1c Measurement on Hospital Readmission Rates: Analysis of 70,000 Clinical Database Patient Records.